# Runbook 4.4.b: Log analysis  (MODEL SOLUTION, instructor copy, do not distribute)

**Skill:** Detect Attacks · **Maps to:** 4.4.B · **Language:** Python 3 (pytest)

**Steps:** 1 Brief · 2 Plan · 3 Build · 4 Test · 5 Try it on the evidence · 6 Wrap up

## 1. Brief

**The situation.** An authentication log records every login attempt as an event with a username, a timestamp (in
seconds), and a result (`"SUCCESS"` or `"FAILED"`). A brute-force attack shows up as a cluster of
failed attempts for the same user, tightly packed in time. Build the correlator that scans the
log and flags every username showing this pattern.

**Lesson objective.** Explain what a security log records and why it matters, identify the common log sources security analysts review, and detect a brute-force login pattern by correlating multiple log entries over time.

### What comes in
A list of dicts, each with keys `ts` (int, seconds since some reference point), `user` (str),
`result` (str, `"SUCCESS"` or `"FAILED"`). Entries are not guaranteed to already be sorted by
timestamp.

### What must come out
A sorted list of usernames (str) flagged for a brute-force pattern.

- [ ] Read the situation and say in one sentence what the tool must decide
- [ ] Find the input and the output above

## 2. Plan

Work it out on paper before you write code.

### Worked examples
```
[{"ts": 0, "user": "jsmith", "result": "FAILED"},
 {"ts": 13, "user": "jsmith", "result": "FAILED"},
 {"ts": 21, "user": "jsmith", "result": "FAILED"},
 {"ts": 30, "user": "jsmith", "result": "FAILED"},
 {"ts": 38, "user": "jsmith", "result": "FAILED"},
 {"ts": 46, "user": "jsmith", "result": "SUCCESS"},
 {"ts": 100, "user": "agarcia", "result": "FAILED"},
 {"ts": 1662, "user": "agarcia", "result": "FAILED"}]
    -> ["jsmith"]
```
*(jsmith has 5 failures spanning 0-38s = 38s <= 60s -> flagged. agarcia has only 2 failures,
1562s apart -> not flagged.)*

- [ ] Work the first example by hand and get the same answer

### Rules to respect
- [ ] Deterministic, offline, standard library only.
- [ ] Input entries may arrive out of timestamp order; sort within your implementation.
- [ ] Return a de-duplicated, alphabetically sorted list of usernames.
- [ ] Must pass all cases in the tests (visible + hidden).
- [ ] Write the steps your function will take, in plain words
- [ ] List two edge cases (an empty input, a tricky value)

<details><summary>Reference: the exact contract</summary>

Any implementation that satisfies this interface can be swapped in (reference, student, or
composite) and must pass the tests unchanged.

#### Entry point
```python
def solve(entries: list) -> list
```

#### Input (`entries`)
A list of dicts, each an authentication log event:

| key | type | meaning |
|---|---|---|
| `ts` | int | timestamp in seconds (not guaranteed sorted across the list) |
| `user` | str | username the attempt was made against |
| `result` | str | `"SUCCESS"` or `"FAILED"` |

#### Output (`result`)
A de-duplicated, alphabetically sorted list of `str` usernames flagged for a brute-force pattern.

#### Detection rule
For each user, collect all `FAILED` timestamps and sort them ascending. The user is flagged if
any 5 consecutive timestamps in that sorted list (`t[i]` through `t[i+4]`) satisfy
`t[i+4] - t[i] <= 60`. A user needs at least 5 total failed attempts to be eligible; `SUCCESS`
entries are ignored entirely for this rule (they do not reset or interrupt a failure streak).

#### Guarantees / rules
- Pure function, deterministic, no I/O, no network.
- Standard library only unless the hidden grading tests: allow_deps` lists otherwise.
- Input order is not guaranteed; sort per-user failed timestamps before evaluating the rule.
- The 60-second window is inclusive (`<= 60`, not `< 60`).

#### Acceptance
Your code passes when every test below shows a checkmark (some are hidden until you run them).

</details>

## 3. Build it

This is the model solution.

- [ ] `solve` handles the worked examples
- [ ] `solve` handles your edge cases

In [ ]:
WINDOW_SECONDS = 60
MIN_FAILURES = 5


def solve(entries: list) -> list:
    failures_by_user = {}
    for entry in entries:
        if entry.get("result") == "FAILED":
            user = entry["user"]
            failures_by_user.setdefault(user, []).append(entry["ts"])

    flagged = set()
    for user, timestamps in failures_by_user.items():
        timestamps.sort()
        if len(timestamps) < MIN_FAILURES:
            continue
        for i in range(len(timestamps) - MIN_FAILURES + 1):
            span = timestamps[i + MIN_FAILURES - 1] - timestamps[i]
            if span <= WINDOW_SECONDS:
                flagged.add(user)
                break

    return sorted(flagged)


## 4. Test it

The tests are the acceptance check, like a change review before a fix goes live.

- [ ] Run the tests
- [ ] Every test says PASS (if not, read the message, go back to step 3)

In [ ]:
import __main__ as impl

def test_example_1_prompt_scenario():
    entries = [
        {"ts": 0, "user": "jsmith", "result": "FAILED"},
        {"ts": 13, "user": "jsmith", "result": "FAILED"},
        {"ts": 21, "user": "jsmith", "result": "FAILED"},
        {"ts": 30, "user": "jsmith", "result": "FAILED"},
        {"ts": 38, "user": "jsmith", "result": "FAILED"},
        {"ts": 46, "user": "jsmith", "result": "SUCCESS"},
        {"ts": 100, "user": "agarcia", "result": "FAILED"},
        {"ts": 1662, "user": "agarcia", "result": "FAILED"},
    ]
    assert impl.solve(entries) == ["jsmith"]


def test_example_2_no_flags():
    entries = [
        {"ts": 0, "user": "mchen", "result": "SUCCESS"},
        {"ts": 500, "user": "rpatel", "result": "FAILED"},
        {"ts": 1700, "user": "rpatel", "result": "FAILED"},
    ]
    assert impl.solve(entries) == []


def test_edge_empty_entries():
    assert impl.solve([]) == []


def test_fewer_than_five_failures_never_flagged():
    entries = [
        {"ts": t, "user": "twilson", "result": "FAILED"} for t in [0, 5, 10, 15]
    ]
    assert impl.solve(entries) == []


def test_five_failures_exactly_at_60s_boundary_flags():
    entries = [
        {"ts": t, "user": "bwright", "result": "FAILED"} for t in [0, 15, 30, 45, 60]
    ]
    assert impl.solve(entries) == ["bwright"]


def test_five_failures_just_over_60s_does_not_flag():
    entries = [
        {"ts": t, "user": "bwright", "result": "FAILED"} for t in [0, 15, 30, 45, 61]
    ]
    assert impl.solve(entries) == []


def test_out_of_order_input_still_correlated_correctly():
    entries = [
        {"ts": 46, "user": "jsmith", "result": "SUCCESS"},
        {"ts": 21, "user": "jsmith", "result": "FAILED"},
        {"ts": 0, "user": "jsmith", "result": "FAILED"},
        {"ts": 38, "user": "jsmith", "result": "FAILED"},
        {"ts": 13, "user": "jsmith", "result": "FAILED"},
        {"ts": 30, "user": "jsmith", "result": "FAILED"},
    ]
    assert impl.solve(entries) == ["jsmith"]


def test_sliding_window_finds_cluster_not_just_first_and_last():
    entries = [
        {"ts": 0, "user": "cluster", "result": "FAILED"},
        {"ts": 100, "user": "cluster", "result": "FAILED"},
        {"ts": 110, "user": "cluster", "result": "FAILED"},
        {"ts": 120, "user": "cluster", "result": "FAILED"},
        {"ts": 130, "user": "cluster", "result": "FAILED"},
        {"ts": 140, "user": "cluster", "result": "FAILED"},
        {"ts": 200, "user": "cluster", "result": "FAILED"},
    ]
    assert impl.solve(entries) == ["cluster"]


def test_multiple_users_result_sorted_alphabetically():
    entries = []
    for t in [0, 10, 20, 30, 40]:
        entries.append({"ts": t, "user": "zach", "result": "FAILED"})
    for t in [0, 10, 20, 30, 40]:
        entries.append({"ts": t, "user": "amy", "result": "FAILED"})
    assert impl.solve(entries) == ["amy", "zach"]


def test_success_entries_do_not_count_as_failures():
    entries = [
        {"ts": 0, "user": "user1", "result": "SUCCESS"},
        {"ts": 5, "user": "user1", "result": "FAILED"},
        {"ts": 10, "user": "user1", "result": "SUCCESS"},
        {"ts": 15, "user": "user1", "result": "FAILED"},
        {"ts": 20, "user": "user1", "result": "SUCCESS"},
        {"ts": 25, "user": "user1", "result": "FAILED"},
    ]
    assert impl.solve(entries) == []


# ---- run every test above ------------------------------------------------
_results = []
for _name, _fn in list(globals().items()):
    if _name.startswith("test_") and callable(_fn):
        try:
            _fn()
            _results.append((_name, None))
        except Exception as _e:
            _results.append((_name, _e))
for _name, _e in _results:
    print(("PASS  " if _e is None else "FAIL  ") + _name + ("" if _e is None else f": {_e!r}"))
print(f"{sum(e is None for _, e in _results)}/{len(_results)} tests passed")


## 5. Try it on the evidence

Run your finished tool on the examples from the brief, then change one value and predict the answer before you run it again.

- [ ] Every example says OK
- [ ] Your own changed example gives the answer you predicted

In [ ]:
checks = [
    (lambda: solve([{"ts": 0, "user": "jsmith", "result": "FAILED"},
 {"ts": 13, "user": "jsmith", "result": "FAILED"},
 {"ts": 21, "user": "jsmith", "result": "FAILED"},
 {"ts": 30, "user": "jsmith", "result": "FAILED"},
 {"ts": 38, "user": "jsmith", "result": "FAILED"},
 {"ts": 46, "user": "jsmith", "result": "SUCCESS"},
 {"ts": 100, "user": "agarcia", "result": "FAILED"},
 {"ts": 1662, "user": "agarcia", "result": "FAILED"}]),
     ["jsmith"]),
]

for n, (run, want) in enumerate(checks, 1):
    try:
        got = run()
    except Exception as e:
        got = "error: " + repr(e)
    print("example", n, "OK  " if got == want else "DIFF", "got:", got, " expected:", want)


## 6. Wrap up

- [ ] Explain in one sentence why your rule catches the problem
- [ ] Name one way an attacker could slip past it (a false negative)
- [ ] Name one harmless input it might wrongly flag (a false positive)
- [ ] Say where this check would run in a real organisation, and who would act on it